<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB09_Clustering_and_Dimensionality_Reduction_with_Real_Data_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB09 · Clase 9 — Agrupamiento y reducción de dimensionalidad con datos reales**

## Bloque 2: IA — Machine Learning (continuación)

Todos los modelos de `NB07`/`NB08` eran **supervisados**: siempre teníamos un objetivo conocido (tipo de combustible, consumo de combustible, mina o roca) que predecir. Esta clase pasa al **aprendizaje no supervisado**, donde no hay ningún objetivo en absoluto — solo el objetivo de encontrar estructura en los datos por sí mismos. Cubrimos dos de las técnicas no supervisadas más utilizadas:

- **Agrupamiento con K-Means (K-Means clustering)**, aplicado al dataset real [`ship_fuel_efficiency.csv`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/ship_fuel_efficiency.csv) de `NB07`, para descubrir "perfiles operativos" de viaje sin usar ninguna de sus etiquetas.
- **Análisis de Componentes Principales (PCA)**, aplicado al dataset real [`sonar.all-data`](https://github.com/JuanZapa7a/AINavalEngineering/blob/main/Datasets/sonar.all-data) de `NB08` (¡60 características!), para visualizar datos de alta dimensión en 2D y ver si el agrupamiento por sí solo puede recuperar la separación real mina/roca.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar qué hace que un problema de aprendizaje sea no supervisado, y cuándo recurrir a él en vez de a un modelo supervisado.
- Explicar cómo K-Means asigna puntos a clústeres y actualiza los centroides, y su principal limitación (hay que elegir *k*).
- Usar el método del codo y el índice de silueta para elegir un número razonable de clústeres.
- Perfilar e interpretar clústeres comparándolos con etiquetas conocidas (pero no usadas).
- Explicar qué hace el PCA (proyecciones que maximizan la varianza) y leer un gráfico de varianza explicada.
- Usar PCA para visualizar datos de alta dimensión en 2D, y evaluar la calidad del agrupamiento frente a la verdad de referencia con el Índice de Rand Ajustado.

### Agenda (clase de 2 horas)

| # | Segmento de la clase | Duración aprox. | Tipo |
|---|---------------------|:---:|:---:|
| 1 | Repaso de NB01–NB08, hoja de ruta de hoy | 5 min | Teoría |
| 2 | Aprendizaje no supervisado: qué es y por qué | 10 min | Teoría |
| 3 | K-Means: cómo funciona | 15 min | Teoría |
| 4 | Práctica: elegir *k* para datos reales de viaje (método del codo, índice de silueta) | 15 min | Práctica |
| 5 | Interpretar clústeres: perfilado y comparación con etiquetas conocidas | 20 min | Práctica |
| 6 | ¿Por qué reducir la dimensionalidad? | 10 min | Teoría |
| 7 | PCA: cómo funciona | 15 min | Teoría |
| 8 | Práctica: PCA + agrupamiento sobre datos reales de sonar con 60 características | 20 min | Práctica |
| 9 | Aprendizaje no supervisado en ingeniería naval/oceánica (panorama) | 5 min | Teoría |
| 10 | Resumen, tarea, próxima clase | 5 min | Teoría |

> Los tiempos son orientación aproximada, no un guion estricto — no hay descansos programados. Si cubrimos todo con tiempo de sobra, la clase termina antes; eso puede pasar y no hay problema.

---

## 1. Repaso: dónde estamos

- **`NB01`–`NB02`**: historia de la IA, fundamentos de Python/Colab/NumPy/Pandas.
- **`NB07`**: el flujo de trabajo de ML supervisado — train/val/test, métricas, un clasificador y un regresor, validación cruzada, fuga de datos.
- **`NB08`**: algoritmos de clasificación en profundidad — árboles de decisión, ensembles, SVM, pipelines a prueba de fugas, búsqueda de hiperparámetros.
- **`NB09`** (hoy): aprendizaje no supervisado — encontrar estructura en los datos **sin ninguna etiqueta objetivo**.

Seguimos dentro del **Bloque 2 — IA: Machine Learning** de la hoja de ruta del curso.

---

## 2. Aprendizaje no supervisado: qué es y por qué

Todos los modelos hasta ahora se han entrenado con ejemplos **etiquetados**: siempre sabíamos la "respuesta correcta" (tipo de combustible, consumo de combustible, mina o roca) para cada fila de entrenamiento. En muchas situaciones reales, **no hay ninguna etiqueta en absoluto** — `bien porque nadie ha anotado los datos, bien porque todavía no sabemos qué categorías existen`. El aprendizaje no supervisado busca estructura en los propios datos:

| Tarea | Objetivo | Ejemplo naval/oceánico |
|---|---|---|
| **Agrupamiento (clustering)** | Agrupar registros similares | Segmentar una flota en perfiles operativos sin categorías predefinidas |
| **Reducción de dimensionalidad** | Comprimir muchas características en menos, conservando la mayor parte de la información | Reducir 60 bandas de frecuencia de sonar a 2 para visualización |
| **Detección de anomalías** | Señalar registros que no encajan en ningún patrón normal | Detectar una lectura inusual del motor antes de que se convierta en una avería |

> **Para saber más**: [Aprendizaje no supervisado (Wikipedia)](https://en.wikipedia.org/wiki/Unsupervised_learning).

Hoy cubrimos las dos primeras — agrupamiento (K-Means) y reducción de dimensionalidad (PCA) — ambas aplicadas a datasets reales que ya has visto.

---

## 3. K-Means: cómo funciona

**K-Means** divide los datos en *k* clústeres, donde *k* se elige de antemano. Funciona de forma iterativa:

1. Coloca *k* centros de clúster iniciales ("centroides"), normalmente en puntos aleatorios.
2. **Asigna** cada punto de datos a su centroide más cercano (por distancia euclídea).
3. **Actualiza** cada centroide a la media de los puntos que tiene asignados ahora.
4. Repite los pasos 2–3 hasta que las asignaciones dejan de cambiar (convergencia).

Dos cosas a tener en cuenta antes de usarlo:
- **Hay que elegir *k* de antemano** — a diferencia del aprendizaje supervisado, no hay un número "correcto" de clústeres escrito en los datos; tenemos que estimar uno razonable (Parte 4).
- **Se basa en distancias, así que la escala importa** — exactamente igual que el SVM en `NB08`, las características en escalas distintas dominarán el cálculo de distancia a menos que se estandaricen primero.

> **Para saber más**: [Agrupamiento K-Means (Wikipedia)](https://en.wikipedia.org/wiki/K-means_clustering) · [documentación de `sklearn.cluster.KMeans`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).

Volvemos al dataset real `ship_fuel_efficiency.csv` de `NB07` — pero esta vez, **ignoramos deliberadamente sus etiquetas** (`ship_type`, `fuel_type`, `weather_conditions`) y agrupamos los viajes usando solo tres características numéricas operativas: `distance`, `fuel_consumption` y `engine_efficiency`. (`CO2_emissions` se deja fuera por la misma razón que en `NB07` — es casi un duplicado de `fuel_consumption`, así que incluirla solo contaría dos veces la misma información en vez de añadir una dimensión nueva.)

In [ ]:
!wget -q -O ship_fuel_efficiency.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/ship_fuel_efficiency.csv

import pandas as pd

fuel = pd.read_csv("ship_fuel_efficiency.csv")
fuel.head()

Escala las tres características de agrupamiento — necesario para K-Means, exactamente igual que para el SVM:

In [ ]:
from sklearn.preprocessing import StandardScaler

cluster_features = ["distance", "fuel_consumption", "engine_efficiency"]
X_cluster = fuel[cluster_features]

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

Ahora la pregunta clave: **¿cuántos clústeres?** Dos herramientas habituales:

- El **método del codo**: representa `inertia_` de K-Means (la distancia al cuadrado total de cada punto a su centroide asignado) frente a *k*. La inercia siempre disminuye a medida que crece *k* — buscamos el punto en el que añadir otro clúster deja de ayudar mucho, formando un "codo" en la curva.
- El **índice de silueta**: mide lo bien separados que están los clústeres (de -1 a 1; cuanto más alto, mejor). A diferencia de la inercia, no favorece automáticamente más clústeres, así que es una segunda opinión útil.

> **Para saber más**: [método del codo (Wikipedia)](https://en.wikipedia.org/wiki/Elbow_method_%28clustering%29) · [silueta (Wikipedia)](https://en.wikipedia.org/wiki/Silhouette_%28clustering%29) · [documentación de `sklearn.metrics.silhouette_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

inertias = []
sil_scores = []
k_range = range(1, 9)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, labels) if k > 1 else None)

Represéntalos lado a lado:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_xlabel("k (number of clusters)")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow method")

axes[1].plot(list(k_range)[1:], sil_scores[1:], marker="o", color="darkorange")
axes[1].set_xlabel("k (number of clusters)")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette score by k")

plt.tight_layout()
plt.show()

**Lee tus propios gráficos**: ¿dónde se dobla visiblemente la curva del codo? ¿El índice de silueta alcanza su máximo en el mismo *k*, o en otro distinto? Los dos métodos no siempre coinciden — cuando no lo hacen, el conocimiento del dominio (¿cuántos perfiles operativos genuinamente distintos tendrían sentido para una flota real?) debería ayudarte a decidir. Para el resto de esta sección usaremos **k = 3**, pero cámbialo y vuelve a ejecutar todo lo que sigue si tus gráficos sugieren otra cosa.

---

## 5. Interpretar clústeres

Ajusta el modelo final y añade la etiqueta de clúster de vuelta al DataFrame:

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
fuel["cluster"] = kmeans.fit_predict(X_cluster_scaled)

fuel["cluster"].value_counts().sort_index()

Un clúster solo es útil una vez que sabemos qué *significa*. Perfila cada clúster por sus valores medios de características — `aquí es donde el conocimiento del dominio convierte "clúster 0, 1, 2" en una historia operativa real`:

In [ ]:
fuel.groupby("cluster")[cluster_features].mean().round(1)

Ver los clústeres directamente sobre dos de las tres características reales hace que la tabla de perfiles sea algo concreto:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(fuel["distance"], fuel["fuel_consumption"], c=fuel["cluster"], cmap="viridis", alpha=0.6, s=15)
ax.set_xlabel("Distance (nm)")
ax.set_ylabel("Fuel consumption (L)")
ax.set_title("Real voyages, colored by K-Means cluster (k=3)")
legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend1)
plt.show()


**Pruébalo tú mismo**: el índice de silueta medio (usado para elegir *k* en la Parte 4) oculta qué puntos *individuales* están en el límite. Calcula `sklearn.metrics.silhouette_samples` para este agrupamiento con `k=3` y encuentra los 5 viajes reales con la silueta por punto más baja — los puntos más cercanos a estar igual de bien explicados por un clúster vecino.

In [ ]:
from sklearn.metrics import silhouette_samples

fuel["silhouette"] = silhouette_samples(X_cluster_scaled, fuel["cluster"])
borderline = fuel.sort_values("silhouette").head(5)
borderline[["cluster", "silhouette"] + cluster_features]


Ahora la prueba real: hemos agrupado **sin usar `ship_type` en ningún momento**. ¿El agrupamiento no supervisado coincide con los tipos de buque conocidos, o ha descubierto algo distinto (p. ej., agrupado por duración/intensidad del viaje en vez de por categoría de buque)? Una tabla de contingencia responde a esto directamente:

In [ ]:
pd.crosstab(fuel["cluster"], fuel["ship_type"])

Si cada clúster está dominado por uno o dos tipos de buque, K-Means ha redescubierto en gran medida una categoría que ya teníamos — una comprobación de cordura tranquilizadora. Si los clústeres mezclan tipos de buque libremente, ha encontrado un patrón *distinto*, igual de válido (p. ej., "viajes cortos y eficientes" frente a "viajes largos e ineficientes") que atraviesa el tipo de buque. Ningún resultado es incorrecto — el aprendizaje no supervisado encuentra la estructura más fuerte que haya en las características que le has dado, que puede coincidir o no con la categoría que tenías en mente.

---

## 6. ¿Por qué reducir la dimensionalidad?

Nuestro ejemplo de agrupamiento usaba 3 características — fáciles de razonar e incluso de representar directamente. El dataset Sonar de `NB08` tiene **60**. Problemas de los datos de alta dimensión:

- **No se pueden visualizar directamente** — los humanos vemos en 2 o 3 dimensiones, no en 60.
- **La maldición de la dimensionalidad**: a medida que crecen las dimensiones, los puntos de datos se vuelven dispersos y los métodos basados en distancias (K-Means, KNN, SVM) se vuelven menos fiables — la mayoría de los puntos terminan siendo aproximadamente equidistantes entre sí.
- **Redundancia y ruido**: las bandas de frecuencia contiguas en un barrido de un sensor suelen estar correlacionadas (como `fuel_consumption`/`CO2_emissions` en `NB07`); buena parte de la "información" en 60 columnas puede en realidad necesitar solo un puñado de dimensiones independientes para capturarse.

> **Para saber más**: [maldición de la dimensionalidad (Wikipedia)](https://en.wikipedia.org/wiki/Curse_of_dimensionality).

La **reducción de dimensionalidad** aborda las tres cosas: `comprime muchas características correlacionadas en menos, sin correlación entre ellas, que conservan tanta información original` (varianza) como sea posible.

---

## 7. PCA: cómo funciona

El **Análisis de Componentes Principales (PCA)** encuentra nuevos ejes ("componentes principales") que son:
1. **Combinaciones lineales** de las características originales,
2. **No correlacionados** entre sí, y
3. Ordenados de forma que el **primer componente captura la mayor varianza** (dispersión) de los datos, el segundo captura la mayor varianza *restante*, y así sucesivamente.

Proyectar los datos sobre solo los primeros 2–3 componentes suele conservar la mayor parte de la estructura significativa, incluso cuando los datos originales tienen docenas de características — porque `las características del mundo real raramente son todas independientes; el PCA encuentra y explota esa redundancia automáticamente`.

Cada componente reporta una **razón de varianza explicada**: la fracción de la varianza total que representa. Representar la suma acumulada indica cuántos componentes habría que conservar para preservar, por ejemplo, el 90% de la información original.

> **Para saber más**: [Análisis de Componentes Principales (Wikipedia)](https://en.wikipedia.org/wiki/Principal_component_analysis) · [documentación de `sklearn.decomposition.PCA`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html).

---

## 8. Práctica: PCA y agrupamiento sobre datos reales de sonar

Vuelve a cargar el dataset real de Sonar (Minas vs. Rocas) de `NB08` — 208 ecos de sonar, 60 características de bandas de frecuencia:

In [ ]:
!wget -q -O sonar.csv https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data

sonar = pd.read_csv("sonar.csv", header=None)
sonar.columns = [f"freq_{i}" for i in range(60)] + ["label"]

X_sonar = sonar.drop(columns="label")
y_sonar = sonar["label"]  # kept aside only to check our unsupervised result later

Escala, y después reduce a 2 componentes únicamente para visualización:

In [ ]:
from sklearn.decomposition import PCA

X_sonar_scaled = StandardScaler().fit_transform(X_sonar)

pca_2d = PCA(n_components=2, random_state=42)
X_sonar_pca2 = pca_2d.fit_transform(X_sonar_scaled)

print("Explained variance ratio (2 components):", pca_2d.explained_variance_ratio_.round(3))
print("Total variance captured:", pca_2d.explained_variance_ratio_.sum().round(3))

**Pruébalo tú mismo**: `pca_2d.components_[0]` contiene las cargas (loadings) de PC1 — cuánto contribuye cada una de las 60 bandas de frecuencia originales a ese primer eje, el más informativo. ¿Qué bandas de frecuencia reales dominan PC1?

In [ ]:
loadings = pd.Series(pca_2d.components_[0], index=X_sonar.columns).sort_values(key=abs, ascending=False)
print("Top 5 contributors to PC1 (by absolute loading):")
print(loadings.head(5))


Dos componentes no van a capturar *toda* la varianza original de 60 características — es de esperar. Veamos cuántos componentes necesitaríamos para la mayor parte:

In [ ]:
pca_full = PCA(random_state=42).fit(X_sonar_scaled)
cumulative_variance = pca_full.explained_variance_ratio_.cumsum()

plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker=".")
plt.axhline(0.9, color="red", linestyle="--", label="90% variance")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA — how many components do we actually need?")
plt.legend()
plt.show()

Ahora visualiza la proyección 2D, coloreada por la etiqueta **real** — recuerda, el propio PCA nunca ve `label`, solo ve las 60 columnas de características:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for lab, name in [("M", "Mine"), ("R", "Rock")]:
    mask = y_sonar == lab
    ax.scatter(X_sonar_pca2[mask, 0], X_sonar_pca2[mask, 1], label=name, alpha=0.6)
ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.set_title("Sonar data in 2D (PCA), colored by true label")
ax.legend()
plt.show()

Si minas y rocas forman dos regiones visualmente separables incluso en esta vista comprimida en 2D, es una buena señal de que los clasificadores de `NB08` tenían estructura real con la que trabajar. Ahora la prueba no supervisada real: agrupa los datos (escalados, con las 60 características completas) con K-Means en 2 grupos, **sin mostrarle nunca las etiquetas reales**, y observa cómo de bien coinciden los clústeres con la realidad usando el **Índice de Rand Ajustado** (1.0 = concordancia perfecta, 0.0 = no mejor que el azar):

In [ ]:
from sklearn.metrics import adjusted_rand_score

kmeans_sonar = KMeans(n_clusters=2, n_init=10, random_state=42)
sonar_clusters = kmeans_sonar.fit_predict(X_sonar_scaled)

ari = adjusted_rand_score(y_sonar, sonar_clusters)
print("Adjusted Rand Index vs. true mine/rock label:", round(ari, 3))
pd.crosstab(sonar_clusters, y_sonar)

Ver la asignación de clústeres de K-Means representada de la misma forma que el gráfico de etiquetas reales anterior hace que el Índice de Rand Ajustado sea algo concreto — dos colores que coinciden en su mayoría significarían un ARI alto; dos colores que dividen los datos de forma distinta significan uno bajo:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_sonar_pca2[:, 0], X_sonar_pca2[:, 1], c=sonar_clusters, cmap="coolwarm", alpha=0.6)
ax.set_xlabel("Principal component 1")
ax.set_ylabel("Principal component 2")
ax.set_title(f"Same 2D projection, colored by K-Means cluster (ARI = {ari:.3f})")
plt.show()


**Interpreta tu propia puntuación**: un ARI alto significaría que el patrón más fuerte en la señal de sonar en bruto coincide con "mina vs. roca" — `genuinamente útil de saber, ya que sugeriría que incluso un método simple y sin etiquetas captura la mayor parte de la señal`. Un ARI bajo significa que el agrupamiento natural más fuerte en los datos es *otra cosa* (p. ej., ángulo de incidencia, intensidad de la señal) — que es exactamente por qué los clasificadores *supervisados* de `NB08`, entrenados específicamente sobre la etiqueta mina/roca, eran la herramienta adecuada para esa tarea. Comparar los dos no es un fallo de ninguno de los métodos — muestra por qué se elige aprendizaje supervisado o no supervisado según si tienes (y confías en) una etiqueta que merezca la pena optimizar.

> **Para saber más**: [Índice de Rand / Índice de Rand Ajustado (Wikipedia)](https://en.wikipedia.org/wiki/Rand_index) · [documentación de `sklearn.metrics.adjusted_rand_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html).

---

## 9. Aprendizaje no supervisado en ingeniería naval y oceánica (panorama)

| Técnica | Ejemplo naval/oceánico |
|---|---|
| Agrupamiento | Segmentar una flota en perfiles operativos; agrupar viajes por comportamiento de ruta/meteorología; identificar regímenes de estado de la mar distintos a partir de datos de boyas |
| Reducción de dimensionalidad | Comprimir muchos canales de sensor correlacionados (vibración, temperatura, presión) antes de pasarlos a un modelo de monitorización; visualizar datos de prospección o sonar de alta dimensión |
| Detección de anomalías (basada en ambas) | Señalar un viaje o una lectura de sensor que no pertenece a ningún clúster conocido, o que se reconstruye mal tras la compresión con PCA — una señal de alerta temprana antes de disponer de un modelo supervisado completo de predicción de fallos |

---

## Resumen de la clase

- El aprendizaje no supervisado encuentra estructura sin etiqueta objetivo — el agrupamiento agrupa registros similares; la reducción de dimensionalidad comprime características correlacionadas.
- K-Means alterna entre asignar puntos al centroide más cercano y recalcular los centroides; hay que elegir *k*, y las características deben escalarse primero.
- El método del codo y el índice de silueta ayudan a elegir *k*, pero el conocimiento del dominio suele tener la última palabra.
- Un clúster solo es útil una vez perfilado e interpretado — comparar los clústeres con una etiqueta conocida pero no usada es una buena comprobación de cordura, coincidan o no al final.
- El PCA encuentra ejes no correlacionados ordenados por cuánta varianza explican, permitiéndonos visualizar en 2D los datos de sonar de 60 dimensiones y entender cuánta información captura realmente un puñado de componentes.
- Los resultados supervisados y no supervisados pueden discrepar (como probablemente mostró nuestro Índice de Rand Ajustado del sonar) — eso es informativo, no un fallo, y es exactamente por qué existen ambos enfoques.

## Para la próxima clase (NB10)

Uniremos todo el flujo de trabajo — curación de datos, selección de características, pipelines, ajuste de hiperparámetros y evaluación final — en un proyecto de Machine Learning completo y ajustado sobre un nuevo dataset real, cerrando el Bloque 2 antes de que empiece el Bloque 3 (Deep Learning).

## Tarea / Ideas de práctica

1. Repite las Partes 4–5 con un *k* distinto (prueba tanto un valor menor como uno mayor que el elegido en clase) — ¿cómo cambia la tabla de perfiles de clústeres de la Parte 5?
2. Repite el agrupamiento de la Parte 4, pero vuelve a añadir `CO2_emissions` a `cluster_features` a pesar del aviso de redundancia tipo fuga — ¿cambia de forma apreciable los clústeres encontrados? ¿Por qué sí o por qué no, dado lo correlacionada que está con `fuel_consumption`?
3. En la Parte 8, prueba `n_components=3` en vez de 2 para la visualización PCA del sonar (necesitarás un gráfico 3D, o tres gráficos 2D por pares) — ¿separa la dimensión extra minas y rocas visiblemente mejor?
4. Agrupa los datos de sonar en `k=3` o `k=4` en vez de 2, y haz la tabla de contingencia contra la etiqueta real como hicimos — con más clústeres que clases reales, ¿qué patrones ves?
5. Elige cualquier dataset de una clase anterior (`Naval_Dataset.csv` de `NB02`, los datos de combustible de buques de `NB07`) y aplícale K-Means usando combinaciones de características distintas a las que hemos usado hasta ahora — ¿qué agrupaciones operativas, si alguna, emergen?

> ***Como siempre: un clúster o un componente principal solo es útil cuando puedes explicar, en términos llanos de ingeniería naval, qué representa realmente.***

> Para un tratamiento más profundo, a nivel de libro de texto, de todo lo cubierto en esta secuencia del Bloque 2 (regresión, clasificación, árboles, SVM, agrupamiento, PCA) en un solo lugar, consulta el libro de texto gratuito [*An Introduction to Statistical Learning*](https://www.statlearning.com/) (James, Witten, Hastie & Tibshirani).